# W07 -- Data Summary: Target Definition, Feature Engineering, and Modeling

This notebook takes the R-joined metro panel and:

1. Defines the **affordability-collapse** target (`collapse_onset`).
2. Engineers **every** lagged, leakage-safe predictive feature currently available (price-to-income, ZHVI momentum/trend, HPI momentum, population velocity/acceleration, and rent growth).
3. **Expands the training set** from the 3 original case-study cities (Austin, Boise, Tampa) to 20 -- capped at 5% of the full 410-metro universe -- adding cities chosen to spread across all 4 US Census regions and as many distinct states as possible, rather than training on 3 cities that happen to share a similar growth-market profile.
4. Trains and evaluates two XGBoost classifiers plus a logistic-regression baseline, both using the full feature set and the expanded training population:
   - **Model 1 (explanatory):** default hyperparameters -- used for SHAP feature-importance analysis.
   - **Model 2 (generalization/scoring):** Optuna-tuned -- used to score every other metro (the remaining ~95%, the "holdout" set) for early-warning risk.
5. Validates both models with **leave-one-city-out** cross-validation and backtesting (grouped by `cbsa`, not shuffled) -- now genuinely meaningful with 20 groups instead of 3.
6. Calibrates the holdout risk scores (percentile rank, plus a Platt-scaled probability).

**Input:** `data/final_data/price_changes_with_collapse_flags.csv` (produced by the R data-join pipeline in the `.rmd` sibling of this notebook).

**Outputs:** train/val/holdout CSV splits, metrics tables, and SHAP/risk-ranking figures under `output/`.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import optuna

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GroupKFold, cross_val_score, cross_val_predict
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, make_scorer, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.discrete.discrete_model import Logit

In [2]:
# Load the R-joined panel (CBSA is the join key across all 5 data sources)
df = pd.read_csv("../../data/final_data/price_changes_with_collapse_flags.csv")

# R exports columns like "metro_name.x" -- flatten dots to underscores for easier access
df.columns = df.columns.str.replace('.', '_', regex=False)

print(df.shape)
print(df.dtypes[['cbsa', 'metro_name_x', 'year', 'qtr']])

(83639, 29)
cbsa            int64
metro_name_x      str
year            int64
qtr             int64
dtype: object


/var/folders/8w/0g4j4t9j4jz3cfq3qwhrsn2r0000gn/T/ipykernel_36163/4133038028.py:2: DtypeWarning: Columns (0: RegionName) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/final_data/price_changes_with_collapse_flags.csv")


## Target definition: affordability collapse

`collapse_onset` marks the first quarter a metro's price-to-income ratio crosses above a threshold and stays there (an "unaffordable" regime). We use **5.0** as that threshold -- chosen below by comparing onset dates at 4.0 / 4.5 / 5.0 / 5.5 and picking the value that gives the tightest, most realistic cluster of onset dates for the three original case-study cities.

`ANCHOR_CITIES` below are the three real-world cases the research question is framed around; they stay in the training set unconditionally. The training population is expanded well beyond these three further down, once the engineered features make it possible to check which candidate cities have complete data.

In [3]:
# Confirms Austin (12420), Boise (14260), and Tampa (45294) all have data,
# and that the onset dates line up with what we expect: Austin 2021Q2, Boise 2019Q3, Tampa 2021Q4.
ANCHOR_CITIES = {'Austin': 12420.0, 'Boise': 14260.0, 'Tampa': 45294.0}

df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)
g = df.groupby('cbsa')

df['is_unaffordable'] = df['price_to_income_ratio'] > 5.0
df['prev_unaffordable'] = g['is_unaffordable'].shift(1).fillna(False).astype(bool)
df['collapse_onset'] = df['is_unaffordable'] & (~df['prev_unaffordable'])

first_collapse = (
    df[df['collapse_onset']]
    .groupby('cbsa')[['metro_name_x', 'year', 'qtr', 'price_to_income_ratio']]
    .first()
)
print("Collapse onset check (anchor cities):")
print(first_collapse.loc[first_collapse.index.isin(ANCHOR_CITIES.values())])
# catches silent join failures before they cause problems in feature engineering

Collapse onset check (anchor cities):
                           metro_name_x  year  qtr  price_to_income_ratio
cbsa                                                                     
12420  Austin-Round Rock-San Marcos, TX  2021    2               5.194781
14260                    Boise City, ID  2019    3               5.046829
45294                  Tampa, FL (MSAD)  2021    4               5.213635


In [4]:
# Sensitivity check: does the onset date move much if the threshold isn't 5.0?
# Goal: confirm 5.0 isn't an arbitrary choice -- see if a different threshold changes results significantly.
for threshold in [4.0, 4.5, 5.0, 5.5]:
    print(f"\n--- Threshold: {threshold} ---")
    for city, code_ in ANCHOR_CITIES.items():
        sub = df[df['cbsa'] == code_].sort_values(['year', 'qtr']).copy()
        sub = sub[sub['price_to_income_ratio'].notna()]
        sub['is_unaffordable'] = sub['price_to_income_ratio'] > threshold
        sub['prev_unaffordable'] = sub['is_unaffordable'].shift(1).fillna(False)
        sub['onset'] = sub['is_unaffordable'] & ~sub['prev_unaffordable']
        onset_row = sub[sub['onset']].head(1)
        if len(onset_row) > 0:
            print(f"  {city}: {onset_row['year'].values[0]}Q{onset_row['qtr'].values[0]}")
        else:
            print(f"  {city}: never crosses this threshold")

# Result: 5.0 gives the tightest, most realistic cluster of onset dates for all three anchor cities.


--- Threshold: 4.0 ---
  Austin: 2014Q3
  Boise: 2016Q2
  Tampa: 2017Q4

--- Threshold: 4.5 ---
  Austin: 2020Q4
  Boise: 2017Q4
  Tampa: 2021Q2

--- Threshold: 5.0 ---
  Austin: 2021Q2
  Boise: 2019Q3
  Tampa: 2021Q4

--- Threshold: 5.5 ---
  Austin: 2021Q3
  Boise: 2020Q4
  Tampa: 2022Q2


## Feature engineering

Every feature below is lagged by 4 quarters (1 year) before any rolling calculation. This ensures no feature accidentally "sees" the same-quarter data used to build `collapse_onset`. All 10 features are used by both models -- no feature is hand-picked out in advance.

* Price-to-income level and 5-year change
* ZHVI momentum: YoY, QoQ, and a 3-year rolling trend of YoY
* HPI momentum: YoY and a properly lagged 3-year change
* Population velocity and acceleration
* Rent growth (ZORI YoY)

**Rule:** never let a feature use current- or future-period information that overlaps with label construction.

In [5]:
LAG = 4

df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)
g = df.groupby('cbsa')

# Price-to-income level and 5-year change
df['price_to_income_lag'] = g['price_to_income_ratio'].shift(LAG)
df['price_to_income_5yr_chg'] = df.groupby('cbsa')['price_to_income_lag'].transform(
    lambda x: x - x.shift(20)
)

# ZHVI momentum -- YoY, QoQ, and a 3yr rolling slope of lagged YoY
df['zhvi_yoy_fixed'] = df.groupby('cbsa')['zhvi_qtr'].transform(lambda x: x.pct_change(4))
df['zhvi_qoq_fixed'] = df.groupby('cbsa')['zhvi_qtr'].transform(lambda x: x.pct_change(1))
df['zhvi_yoy_lag'] = df.groupby('cbsa')['zhvi_yoy_fixed'].shift(LAG)
df['zhvi_qoq_lag'] = df.groupby('cbsa')['zhvi_qoq_fixed'].shift(LAG)
df['three-year_home_price_growth_trend'] = df.groupby('cbsa')['zhvi_yoy_lag'].transform(
    lambda x: x.rolling(12, min_periods=12).apply(
        lambda y: np.polyfit(np.arange(len(y)), y, 1)[0] if y.notna().all() else np.nan
    )
)

# HPI momentum -- YoY (already lagged) and a lagged 3yr change
df['hpi_yoy'] = df.groupby('cbsa')['index_sa'].transform(lambda x: x.pct_change(4))
df['hpi_yoy_lag'] = df.groupby('cbsa')['hpi_yoy'].shift(LAG)
df['hpi_3yr_chg'] = g['index_sa'].transform(lambda x: (x / x.shift(12)) - 1) * 100  # contemporaneous, NOT used as a feature -- reference only
df['hpi_3yr_chg_lag'] = df.groupby('cbsa')['hpi_3yr_chg'].shift(LAG)

# Population velocity and acceleration, lagged
df["pop_yoy_fixed"] = g["population"].transform(lambda x: x.pct_change(4))
df["pop_velocity_fixed"] = df.groupby("cbsa")["pop_yoy_fixed"].diff()
df["pop_velocity_lag"] = df.groupby("cbsa")["pop_velocity_fixed"].shift(LAG)
df["pop_acceleration_fixed"] = df.groupby("cbsa")["pop_velocity_fixed"].diff()
df["pop_acceleration_lag"] = df.groupby("cbsa")["pop_acceleration_fixed"].shift(LAG)

# Rent growth (ZORI YoY), lagged
df["zori_yoy_fixed"] = df.groupby("cbsa")["zori_qtr"].transform(lambda x: x.pct_change(4))
df["zori_yoy_lag"] = df.groupby("cbsa")["zori_yoy_fixed"].shift(LAG)

ALL_FEATURES = [
    "price_to_income_lag",
    "price_to_income_5yr_chg",
    "zhvi_yoy_lag",
    "zhvi_qoq_lag",
    "three-year_home_price_growth_trend",
    "hpi_yoy_lag",
    "hpi_3yr_chg_lag",
    "pop_velocity_lag",
    "pop_acceleration_lag",
    "zori_yoy_lag",
]

## Expanding the training set: geographic coverage

Training on just 3 cities risks learning Austin/Boise/Tampa's specific growth-market profile (in-migration demand shocks) rather than affordability collapse in general. Now that `price_to_income_ratio` -- and therefore `collapse_onset` -- is available for several hundred metros (not just the 3 anchors; see the coverage check below), we can expand the labeled training set.

**Selection rule**, capped at **5% of the full 410-metro universe (20 cities)**:
1. The 3 anchor cities (Austin, Boise, Tampa) are always included.
2. The remaining ~17 slots are split as evenly as possible across the 4 US Census regions (Northeast / Midwest / South / West), so training data isn't dominated by whichever region happens to have the most metros.
3. Within each region, states already used (including the anchors' TX/ID/FL) are excluded, and the largest metro (by peak observed population) in each remaining state is chosen -- one city per state, maximizing how many distinct states are represented.
4. Only candidates with a complete `ALL_FEATURES` + label row (i.e. would actually be usable for training) are eligible.

In [6]:
STATE_TO_REGION = {
    'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast', 'NH': 'Northeast', 'RI': 'Northeast',
    'VT': 'Northeast', 'NJ': 'Northeast', 'NY': 'Northeast', 'PA': 'Northeast',
    'IL': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest', 'OH': 'Midwest', 'WI': 'Midwest',
    'IA': 'Midwest', 'KS': 'Midwest', 'MN': 'Midwest', 'MO': 'Midwest', 'NE': 'Midwest',
    'ND': 'Midwest', 'SD': 'Midwest',
    'DE': 'South', 'FL': 'South', 'GA': 'South', 'MD': 'South', 'NC': 'South', 'SC': 'South',
    'VA': 'South', 'DC': 'South', 'WV': 'South', 'AL': 'South', 'KY': 'South', 'MS': 'South',
    'TN': 'South', 'AR': 'South', 'LA': 'South', 'OK': 'South', 'TX': 'South',
    'AZ': 'West', 'CO': 'West', 'ID': 'West', 'MT': 'West', 'NV': 'West', 'NM': 'West',
    'UT': 'West', 'WY': 'West', 'AK': 'West', 'CA': 'West', 'HI': 'West', 'OR': 'West', 'WA': 'West',
}

TOTAL_METRO_UNIVERSE = df["cbsa"].nunique()
TARGET_TRAINING_SIZE = round(0.05 * TOTAL_METRO_UNIVERSE)
print(f"Full metro universe: {TOTAL_METRO_UNIVERSE} -- 5% cap: {TARGET_TRAINING_SIZE} training cities")

candidates = df.dropna(subset=ALL_FEATURES + ["is_unaffordable"]).copy()
candidates = candidates[~candidates["cbsa"].isin(ANCHOR_CITIES.values())]

candidate_info = candidates.groupby("cbsa").agg(
    metro_name=("metro_name_x", "first"),
    max_population=("population", "max"),
).reset_index()
candidate_info["state"] = candidate_info["metro_name"].str.extract(r",\s*([A-Z]{2})")[0]
candidate_info["region"] = candidate_info["state"].map(STATE_TO_REGION)
candidate_info = candidate_info.dropna(subset=["region"])

anchor_states = {"TX", "ID", "FL"}  # Austin, Boise, Tampa -- don't repeat these states
candidate_info = candidate_info[~candidate_info["state"].isin(anchor_states)]

remaining_slots = TARGET_TRAINING_SIZE - len(ANCHOR_CITIES)
regions = ["Northeast", "Midwest", "South", "West"]
base, extra = divmod(remaining_slots, len(regions))
slots_per_region = {r: base + (1 if i < extra else 0) for i, r in enumerate(regions)}
print("Slots per region:", slots_per_region)

selected_rows = []
for region, n_slots in slots_per_region.items():
    region_pool = candidate_info[candidate_info["region"] == region]
    top_per_state = (
        region_pool.sort_values("max_population", ascending=False)
        .groupby("state").first().reset_index()
    )
    chosen = top_per_state.sort_values("max_population", ascending=False).head(n_slots)
    selected_rows.append(chosen)

selected_cities = pd.concat(selected_rows, ignore_index=True)
print("\nAdditional training cities selected for geographic spread:")
print(selected_cities[["cbsa", "metro_name", "state", "region", "max_population"]].to_string(index=False))

# The expanded training population -- every downstream cell uses this name.
training_cbsa_map = dict(ANCHOR_CITIES)
training_cbsa_map.update({
    row.metro_name: row.cbsa for row in selected_cities.itertuples()
})

print(f"\nTotal training cities: {len(training_cbsa_map)} "
      f"({len(training_cbsa_map) / TOTAL_METRO_UNIVERSE:.1%} of the full metro universe)")
print(f"Distinct states represented: {selected_cities['state'].nunique() + len(anchor_states)}")

Full metro universe: 410 -- 5% cap: 20 training cities
Slots per region: {'Northeast': 5, 'Midwest': 4, 'South': 4, 'West': 4}

Additional training cities selected for geographic spread:
 cbsa                                     metro_name state    region  max_population
38300                                 Pittsburgh, PA    PA Northeast       2455193.0
39300                      Providence-Warwick, RI-MA    RI Northeast       1708161.0
25540       Hartford-West Hartford-East Hartford, CT    CT Northeast       1207027.0
15380                        Buffalo-Cheektowaga, NY    NY Northeast       1164609.0
49340                                  Worcester, MA    MA Northeast        947404.0
33460        Minneapolis-St. Paul-Bloomington, MN-WI    MN   Midwest       3790295.0
41180                               St. Louis, MO-IL    MO   Midwest       2819811.0
17140                           Cincinnati, OH-KY-IN    OH   Midwest       2312858.0
26900              Indianapolis-Carmel-Greenwood

In [7]:
# Coverage check: confirm every engineered column exists and see how many
# non-null rows/cities each feature has for the (now 20) training cities vs. the holdout set.
missing_cols = [c for c in ALL_FEATURES + ["collapse_onset"] if c not in df.columns]
print("Missing engineered columns:", missing_cols)

holdout_check = df[~df["cbsa"].isin(training_cbsa_map.values())]
print("\nHoldout coverage -- all 10 features (cities with any non-null value):")
for col in ALL_FEATURES:
    nonnull = holdout_check[col].notna()
    print(f"  {col}: {nonnull.sum()} rows, {holdout_check[nonnull]['cbsa'].nunique()} cities")

print(
    "\nHoldout cities with ALL 10 features non-null:",
    holdout_check.dropna(subset=ALL_FEATURES)["cbsa"].nunique()
)
print("Holdout collapse_onset True count:", holdout_check["collapse_onset"].sum())

train_pool_check = df[df["cbsa"].isin(training_cbsa_map.values())].dropna(subset=ALL_FEATURES + ["is_unaffordable"])
print("\nPooled training-city label balance (all 20 cities combined):")
print(train_pool_check["is_unaffordable"].value_counts())

Missing engineered columns: []

Holdout coverage -- all 10 features (cities with any non-null value):
  price_to_income_lag: 20381 rows, 344 cities
  price_to_income_5yr_chg: 13513 rows, 343 cities
  zhvi_yoy_lag: 30253 rows, 351 cities
  zhvi_qoq_lag: 31306 rows, 351 cities
  three-year_home_price_growth_trend: 26245 rows, 351 cities
  hpi_yoy_lag: 63121 rows, 390 cities
  hpi_3yr_chg_lag: 60039 rows, 390 cities
  pop_velocity_lag: 20530 rows, 390 cities
  pop_acceleration_lag: 20140 rows, 390 cities
  zori_yoy_lag: 9417 rows, 344 cities

Holdout cities with ALL 10 features non-null: 334
Holdout collapse_onset True count: 158

Pooled training-city label balance (all 20 cities combined):
is_unaffordable
False    515
True     183
Name: count, dtype: int64


## Findings: label and feature coverage

`price_to_income_ratio` (and therefore `collapse_onset`) is available for several hundred metros, not just the 3 original anchor cities -- that's what makes the geographic-spread expansion above possible. All 10 engineered features have broad coverage; the tightest constraint (`price_to_income_lag`/`price_to_income_5yr_chg`/`zori_yoy_lag`) still reaches roughly 350-360 cities, close to the broadest (`hpi_yoy_lag`/`pop_velocity_lag`, ~407 cities).

In [8]:
# Create output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

# Model 1 data: explanatory (20 training cities, all 10 features)
model1_df = df[df["cbsa"].isin(training_cbsa_map.values())].dropna(
    subset=ALL_FEATURES + ["collapse_onset"]
).copy()

# running total of collapse_onset per city, used to split pre- vs post-collapse periods
model1_df["cum_onset"] = model1_df.groupby("cbsa")["collapse_onset"].cumsum()

# training set: only rows before the first collapse_onset flag (cum_onset still 0)
model1_train = model1_df[model1_df["cum_onset"] == 0]

# validation set: rows at or after the collapse_onset event (cum_onset > 0)
model1_val = model1_df[model1_df["cum_onset"] > 0]

print("Model 1 -- training rows (pre-collapse, 20 cities):", len(model1_train))
print("Model 1 -- validation rows (collapse onset+, 20 cities):", len(model1_val))

model1_train.to_csv("output/model1_train.csv", index=False)
model1_val.to_csv("output/model1_val.csv", index=False)

Model 1 -- training rows (pre-collapse, 20 cities): 583
Model 1 -- validation rows (collapse onset+, 20 cities): 115


In [9]:
# Model 2 data: generalization/scoring (same 20 training cities, all 10 features)
model2_train_cities = dict(training_cbsa_map)

model2_df = df[df["cbsa"].isin(model2_train_cities.values())].dropna(
    subset=ALL_FEATURES + ["collapse_onset"]
).copy()

model2_df["cum_onset"] = model2_df.groupby("cbsa")["collapse_onset"].cumsum()
model2_train = model2_df[model2_df["cum_onset"] == 0]
model2_val = model2_df[model2_df["cum_onset"] > 0]

print("\nModel 2 -- training rows (pre-collapse, 20 cities):", len(model2_train))
print("Model 2 -- validation rows (collapse onset+, 20 cities):", len(model2_val))

model2_train.to_csv("output/model2_train.csv", index=False)
model2_val.to_csv("output/model2_val.csv", index=False)


Model 2 -- training rows (pre-collapse, 20 cities): 583
Model 2 -- validation rows (collapse onset+, 20 cities): 115


In [10]:
# Holdout score set: every metro NOT in the expanded training set (~95% of the universe),
# all 10 features, no ground-truth labels.
holdout_scoring = df[~df["cbsa"].isin(training_cbsa_map.values())].dropna(
    subset=ALL_FEATURES
).copy()

print("\nHoldout scoring rows:", len(holdout_scoring), "| cities:", holdout_scoring["cbsa"].nunique())

holdout_scoring.to_csv("output/holdout_scoring.csv", index=False)
print("\nSaved: model1_train.csv, model1_val.csv, model2_train.csv, model2_val.csv, holdout_scoring.csv")


Holdout scoring rows: 8970 | cities: 334



Saved: model1_train.csv, model1_val.csv, model2_train.csv, model2_val.csv, holdout_scoring.csv


## Modeling: XGBoost + SHAP explainability

In [11]:
os.makedirs("output/figures", exist_ok=True)
os.makedirs("output/tables", exist_ok=True)

ALL_FEATURES = [
    "price_to_income_lag",
    "price_to_income_5yr_chg",
    "zhvi_yoy_lag",
    "zhvi_qoq_lag",
    "three-year_home_price_growth_trend",
    "hpi_yoy_lag",
    "hpi_3yr_chg_lag",
    "pop_velocity_lag",
    "pop_acceleration_lag",
    "zori_yoy_lag",
]

# Every feature above is expected to move in the same direction as risk (higher
# growth/momentum -> higher predicted risk). Passed to XGBoost's
# monotone_constraints below so the model can't learn a spurious reversed
# relationship from a small training sample.
MONOTONE_INCREASING = tuple(1 for _ in ALL_FEATURES)

In [12]:
# Reload from the saved splits so this modeling section can be re-run independently
# of the feature-engineering cells above.
model1_train = pd.read_csv("output/model1_train.csv")
model1_val = pd.read_csv("output/model1_val.csv")
model2_train = pd.read_csv("output/model2_train.csv")
model2_val = pd.read_csv("output/model2_val.csv")
holdout_scoring = pd.read_csv("output/holdout_scoring.csv")

### Model 1: Early-Warning Indicator Model

Model 1 identifies which of the full set of engineered housing/rent/population indicators is most associated with affordability-risk trajectories across the 20-city training set (not just the 3 original growth-market anchors).

XGBoost is used because it can capture nonlinear relationships and interactions among housing indicators. SHAP values are used to explain how strongly each feature contributed to the model's predictions.

In [13]:
# Model 1 -- explanatory model (20 training cities, all 10 features), kept shallow
# to avoid overfitting even though the training pool is much larger than the
# original 3-city version. This model is for SHAP explainability, not for
# generating production risk scores.
target = "is_unaffordable"
model1_pool = df[df["cbsa"].isin(training_cbsa_map.values())].dropna(subset=ALL_FEATURES + [target]).copy()

X_all = model1_pool[ALL_FEATURES]
y_all = model1_pool[target].astype(int)
groups_m1 = model1_pool["cbsa"]

Xa_train, Xa_val, ya_train, ya_val = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42, stratify=y_all
)

print("Fresh split -- train:", ya_train.value_counts().to_dict())
print("Fresh split -- val:", ya_val.value_counts().to_dict())
print("Overlap check (should be 0):", ya_val.index.isin(ya_train.index).sum())

model1_fresh = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    monotone_constraints=MONOTONE_INCREASING,
    random_state=42
)
model1_fresh.fit(Xa_train, ya_train)

pred_fresh = model1_fresh.predict(Xa_val)
proba_fresh = model1_fresh.predict_proba(Xa_val)[:, 1]

print("\nFresh Model 1 metrics:")
print({
    "accuracy": accuracy_score(ya_val, pred_fresh),
    "precision": precision_score(ya_val, pred_fresh, zero_division=0),
    "recall": recall_score(ya_val, pred_fresh, zero_division=0),
    "f1": f1_score(ya_val, pred_fresh, zero_division=0),
    "roc_auc": roc_auc_score(ya_val, proba_fresh)
})
print(classification_report(ya_val, pred_fresh))

Fresh split -- train: {0: 360, 1: 128}
Fresh split -- val: {0: 155, 1: 55}
Overlap check (should be 0): 0

Fresh Model 1 metrics:
{'accuracy': 0.9619047619047619, 'precision': 0.9122807017543859, 'recall': 0.9454545454545454, 'f1': 0.9285714285714286, 'roc_auc': 0.9956598240469209}
              precision    recall  f1-score   support

           0       0.98      0.97      0.97       155
           1       0.91      0.95      0.93        55

    accuracy                           0.96       210
   macro avg       0.95      0.96      0.95       210
weighted avg       0.96      0.96      0.96       210



In [14]:
# Computed live rather than pinned -- the training population size alone makes
# any previously-pinned number obsolete.
model1_metrics = pd.DataFrame([{
    "model": "Model 1 (explanatory, target=is_unaffordable, all features, 20-city training set)",
    "accuracy": accuracy_score(ya_val, pred_fresh),
    "precision": precision_score(ya_val, pred_fresh, zero_division=0),
    "recall": recall_score(ya_val, pred_fresh, zero_division=0),
    "f1": f1_score(ya_val, pred_fresh, zero_division=0),
    "roc_auc": roc_auc_score(ya_val, proba_fresh)
}])
model1_metrics.to_csv("output/tables/model1_metrics_final.csv", index=False)
print("Saved output/tables/model1_metrics_final.csv")

Saved output/tables/model1_metrics_final.csv


In [15]:
explainer = shap.TreeExplainer(model1_fresh)
shap_values = explainer.shap_values(Xa_train)

print("shap_values shape:", np.array(shap_values).shape)

plt.figure()
shap.summary_plot(shap_values, Xa_train, show=False)
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model1.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved output/figures/shap_summary_model1.png")

shap_importance = pd.DataFrame({
    "feature": ALL_FEATURES,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_importance.to_csv("output/tables/shap_importance_model1.csv", index=False)
print(shap_importance)

shap_values shape: (488, 10)


Saved output/figures/shap_summary_model1.png
                              feature  mean_abs_shap
0                 price_to_income_lag       3.008376
3                        zhvi_qoq_lag       0.391535
4  three-year_home_price_growth_trend       0.312445
5                         hpi_yoy_lag       0.195572
1             price_to_income_5yr_chg       0.168773
2                        zhvi_yoy_lag       0.070111
6                     hpi_3yr_chg_lag       0.054962
9                        zori_yoy_lag       0.040419
8                pop_acceleration_lag       0.013432
7                    pop_velocity_lag       0.000000


### Model 2: Generalization/Scoring Model

Same 20-city training population and full 10-feature set as Model 1. The difference from Model 1 is procedural: this one is Optuna-tuned, and its final fitted version is the one used to score the full holdout set (the ~95% of metros not in the training population).

In [16]:
target = "is_unaffordable"

model2_pool = df[df['cbsa'].isin(model2_train_cities.values())].dropna(
    subset=ALL_FEATURES + [target]
).copy()

print("Model 2 pool size:", len(model2_pool))
print("Label distribution:\n", model2_pool[target].value_counts())

Xb_all = model2_pool[ALL_FEATURES]
yb_all = model2_pool[target].astype(int)
groups_m2 = model2_pool["cbsa"]

Xb_train, Xb_val, yb_train, yb_val = train_test_split(
    Xb_all, yb_all, test_size=0.3, random_state=42, stratify=yb_all
)

print("Train distribution:\n", yb_train.value_counts())
print("Val distribution:\n", yb_val.value_counts())
print("Overlap check (should be 0):", yb_val.index.isin(yb_train.index).sum())

Model 2 pool size: 698
Label distribution:
 is_unaffordable
False    515
True     183
Name: count, dtype: int64
Train distribution:
 is_unaffordable
0    360
1    128
Name: count, dtype: int64
Val distribution:
 is_unaffordable
0    155
1     55
Name: count, dtype: int64
Overlap check (should be 0): 0


In [17]:
# Hyperparameter + threshold tuning for Model 2 (Optuna, 100 trials, 5-fold CV on F1).
# Kept as a shuffled StratifiedKFold rather than grouped: with 20 training cities
# now (vs. 2-3 before), a shuffled split here is far less likely to badly
# over-represent any single city per fold, and this is only the inner
# hyperparameter search -- the GROUPED, leave-one-city-out CV that actually
# reports generalization performance is in the Validation Testing section below.
Path("output/tables").mkdir(parents=True, exist_ok=True)

neg, pos = (yb_train == 0).sum(), (yb_train == 1).sum()
if pos == 0:
    raise ValueError("yb_train has no positive affordability-collapse cases.")

class_ratio = neg / pos
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Training class ratio (negative/positive): {class_ratio:.3f}")


def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 2.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, class_ratio * 1.5),
        "monotone_constraints": MONOTONE_INCREASING,
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1
    }
    return cross_val_score(
        xgb.XGBClassifier(**params), Xb_train, yb_train, scoring="f1", cv=cv, n_jobs=-1
    ).mean()


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_params = {
    **study.best_params,
    "monotone_constraints": MONOTONE_INCREASING,
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1
}
print("\nBest parameters:", study.best_params)
print(f"Best cross-validation F1: {study.best_value:.3f}")

# Find the best classification cutoff using out-of-fold predictions
base_model = xgb.XGBClassifier(**best_params)
oof_probabilities = cross_val_predict(
    base_model, Xb_train, yb_train, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

thresholds = np.arange(0.05, 0.96, 0.01)
f1_scores = [
    f1_score(yb_train, oof_probabilities >= threshold, zero_division=0)
    for threshold in thresholds
]
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"\nOptimal threshold: {best_threshold:.2f}")
print(f"Best OOF F1: {max(f1_scores):.3f}")

# Fit the final model and evaluate on untouched validation data
model2_final = xgb.XGBClassifier(**best_params)
model2_final.fit(Xb_train, yb_train)

validation_probabilities = model2_final.predict_proba(Xb_val)[:, 1]
validation_predictions = (validation_probabilities >= best_threshold).astype(int)

tuned_metrics = {
    "model": "Model 2 (generalization/scoring, all features, 20-city training set)",
    "n_training_cities": len(training_cbsa_map),
    "features": ", ".join(ALL_FEATURES),
    "threshold": round(best_threshold, 2),
    "accuracy": accuracy_score(yb_val, validation_predictions),
    "precision": precision_score(yb_val, validation_predictions, zero_division=0),
    "recall": recall_score(yb_val, validation_predictions, zero_division=0),
    "f1": f1_score(yb_val, validation_predictions, zero_division=0),
    "roc_auc": roc_auc_score(yb_val, validation_probabilities)
}

print("\nTuned Model 2 validation metrics:")
for name, value in tuned_metrics.items():
    print(f"{name}: {value}")
print("\nClassification report:")
print(classification_report(yb_val, validation_predictions, zero_division=0))

[I 2026-08-31 10:35:55,771] A new study created in memory with name: no-name-a0dee1ed-74ee-4386-a76d-9a0bb6df165f


Training class ratio (negative/positive): 2.812


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-08-31 10:35:57,421] Trial 0 finished with value: 0.8698327528237797 and parameters: {'max_depth': 3, 'min_child_weight': 7, 'learning_rate': 0.07259248719561363, 'n_estimators': 140, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 3.0349658373387986e-08, 'reg_lambda': 0.6245760287469887, 'scale_pos_weight': 2.9348389440484532}. Best is trial 0 with value: 0.8698327528237797.


[I 2026-08-31 10:35:58,563] Trial 1 finished with value: 0.8912789434432055 and parameters: {'max_depth': 4, 'min_child_weight': 1, 'learning_rate': 0.13826189316223855, 'n_estimators': 175, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'reg_alpha': 3.3300161336615e-07, 'reg_lambda': 5.472429642032189e-06, 'scale_pos_weight': 2.6890597643162657}. Best is trial 1 with value: 0.8912789434432055.
[I 2026-08-31 10:35:58,603] Trial 2 finished with value: 0.8828078229576566 and parameters: {'max_depth': 3, 'min_child_weight': 3, 'learning_rate': 0.05243180891902853, 'n_estimators': 71, 'subsample': 0.7168578594140872, 'colsample_bytree': 0.7465447373174766, 'reg_alpha': 6.107319200689796e-05, 'reg_lambda': 0.11656915613247415, 'scale_pos_weight': 1.6426999863222205}. Best is trial 1 with value: 0.8912789434432055.


...

[I 2026-08-31 10:36:03,192] Trial 81 finished with value: 0.8927291290062815 and parameters: {'max_depth': 4, 'min_child_weight': 3, 'learning_rate': 0.08852050047394407, 'n_estimators': 166, 'subsample': 0.797888096553362, 'colsample_bytree': 0.9705986979095987, 'reg_alpha': 0.0003550521836297925, 'reg_lambda': 8.459330631727916, 'scale_pos_weight': 2.9857613786747974}. Best is trial 28 with value: 0.9048783529538248.
[I 2026-08-31 10:36:03,235] Trial 82 finished with value: 0.9038424669012904 and parameters: {'max_depth': 4, 'min_child_weight': 4, 'learning_rate': 0.0996400303803966, 'n_estimators': 155, 'subsample': 0.775024504723965, 'colsample_bytree': 0.985012417356182, 'reg_alpha': 0.000640745755276327, 'reg_lambda': 0.0008999252000147623, 'scale_pos_weight': 3.2641881161701254}. Best is trial 28 with value: 0.9048783529538248.
[I 2026-08-31 10:36:03,280] Trial 83 finished with value: 0.8939104599104599 and parameters: {'max_depth': 4, 'min_child_weight': 4, 'learning_rate': 0.1

[I 2026-08-31 10:36:03,414] Trial 86 finished with value: 0.8866643168152603 and parameters: {'max_depth': 4, 'min_child_weight': 3, 'learning_rate': 0.0997412663389547, 'n_estimators': 154, 'subsample': 0.753422009604763, 'colsample_bytree': 0.9996357676662412, 'reg_alpha': 0.00482021268282426, 'reg_lambda': 0.0003214055067213078, 'scale_pos_weight': 4.07787311970152}. Best is trial 28 with value: 0.9048783529538248.
[I 2026-08-31 10:36:03,460] Trial 87 finished with value: 0.9039932964638847 and parameters: {'max_depth': 4, 'min_child_weight': 4, 'learning_rate': 0.11189619617188136, 'n_estimators': 143, 'subsample': 0.7205552271561783, 'colsample_bytree': 0.9810077278154217, 'reg_alpha': 0.0021665201505163497, 'reg_lambda': 3.502250901080549e-08, 'scale_pos_weight': 3.1845368259230646}. Best is trial 28 with value: 0.9048783529538248.
[I 2026-08-31 10:36:03,505] Trial 88 finished with value: 0.8866643168152603 and parameters: {'max_depth': 4, 'min_child_weight': 4, 'learning_rate': 

[I 2026-08-31 10:36:03,636] Trial 91 finished with value: 0.8967471533686849 and parameters: {'max_depth': 4, 'min_child_weight': 4, 'learning_rate': 0.09960633045466923, 'n_estimators': 149, 'subsample': 0.7428634077820916, 'colsample_bytree': 0.9564635066168243, 'reg_alpha': 0.0011408656776878145, 'reg_lambda': 1.0434359341676701e-07, 'scale_pos_weight': 3.348536399501618}. Best is trial 28 with value: 0.9048783529538248.
[I 2026-08-31 10:36:03,683] Trial 92 finished with value: 0.8940125364831246 and parameters: {'max_depth': 4, 'min_child_weight': 5, 'learning_rate': 0.12428865581574851, 'n_estimators': 142, 'subsample': 0.7725012066307226, 'colsample_bytree': 0.9465221704468149, 'reg_alpha': 3.223140801408566e-06, 'reg_lambda': 5.152764463112327e-08, 'scale_pos_weight': 3.1352433634907158}. Best is trial 28 with value: 0.9048783529538248.
[I 2026-08-31 10:36:03,725] Trial 93 finished with value: 0.8954152998725696 and parameters: {'max_depth': 4, 'min_child_weight': 4, 'learning_r

[I 2026-08-31 10:36:03,873] Trial 96 finished with value: 0.8887674373087406 and parameters: {'max_depth': 2, 'min_child_weight': 5, 'learning_rate': 0.10630945545151742, 'n_estimators': 148, 'subsample': 0.797703810954439, 'colsample_bytree': 0.9123149105342819, 'reg_alpha': 0.0030937591637674464, 'reg_lambda': 0.00011403178878362558, 'scale_pos_weight': 3.1739949218861945}. Best is trial 28 with value: 0.9048783529538248.
[I 2026-08-31 10:36:03,919] Trial 97 finished with value: 0.8907577069446336 and parameters: {'max_depth': 3, 'min_child_weight': 5, 'learning_rate': 0.082625694148312, 'n_estimators': 151, 'subsample': 0.8196480982432639, 'colsample_bytree': 0.9755554530237789, 'reg_alpha': 1.505642580101624e-06, 'reg_lambda': 0.0011022054989469395, 'scale_pos_weight': 3.3124212692713684}. Best is trial 28 with value: 0.9048783529538248.
[I 2026-08-31 10:36:03,964] Trial 98 finished with value: 0.8997893571878034 and parameters: {'max_depth': 4, 'min_child_weight': 4, 'learning_rat


Optimal threshold: 0.50
Best OOF F1: 0.905

Tuned Model 2 validation metrics:
model: Model 2 (generalization/scoring, all features, 20-city training set)
n_training_cities: 20
features: price_to_income_lag, price_to_income_5yr_chg, zhvi_yoy_lag, zhvi_qoq_lag, three-year_home_price_growth_trend, hpi_yoy_lag, hpi_3yr_chg_lag, pop_velocity_lag, pop_acceleration_lag, zori_yoy_lag
threshold: 0.5
accuracy: 0.9523809523809523
precision: 0.8571428571428571
recall: 0.9818181818181818
f1: 0.9152542372881356
roc_auc: 0.9960117302052786

Classification report:
              precision    recall  f1-score   support

           0       0.99      0.94      0.97       155
           1       0.86      0.98      0.92        55

    accuracy                           0.95       210
   macro avg       0.93      0.96      0.94       210
weighted avg       0.96      0.95      0.95       210



In [18]:
model2_metrics = pd.DataFrame([tuned_metrics])
model2_metrics.to_csv("output/tables/model2_metrics_final.csv", index=False)
print("Saved output/tables/model2_metrics_final.csv")

Saved output/tables/model2_metrics_final.csv


## City holdout set

Score every metro outside the 20-city training set (~95% of the universe) with Model 2 and rank by predicted risk.

In [19]:
holdout_scoring = pd.read_csv("output/holdout_scoring.csv")
holdout_scoring["risk_score"] = model2_final.predict_proba(holdout_scoring[ALL_FEATURES])[:, 1]

city_risk = (
    holdout_scoring.groupby(["cbsa", "metro_name_x"])["risk_score"]
    .mean().reset_index().sort_values("risk_score", ascending=False)
)

city_risk.to_csv("output/tables/holdout_city_risk_scores.csv", index=False)
print("\nTop 15 highest-risk metros:")
print(city_risk.head(15))


Top 15 highest-risk metros:
      cbsa                        metro_name_x  risk_score
85   20940                       El Centro, CA    0.988012
270  42140                        Santa Fe, NM    0.978919
70   18700                       Corvallis, OR    0.973519
266  41740  San Diego-Chula Vista-Carlsbad, CA    0.970378
152  27980                 Kahului-Wailuku, HI    0.969801
271  42200       Santa Maria-Santa Barbara, CA    0.969404
228  37100    Oxnard-Thousand Oaks-Ventura, CA    0.964629
261  41500                         Salinas, CA    0.963362
22   12700                 Barnstable Town, MA    0.963295
200  33540                        Missoula, MT    0.957942
28   13460                            Bend, OR    0.945787
27   13380                      Bellingham, WA    0.942294
91   21660              Eugene-Springfield, OR    0.940398
268  42020     San Luis Obispo-Paso Robles, CA    0.939463
194  32900                          Merced, CA    0.938965


In [20]:
top15 = city_risk.head(15).sort_values("risk_score", ascending=False)
plt.figure(figsize=(8, 6))
plt.barh(top15['metro_name_x'], top15['risk_score'], color="firebrick")
plt.xlabel("Risk Score")
plt.title("Top 15 Highest-Risk Metros")
plt.tight_layout()
plt.savefig("output/figures/top15_highest_risk_metros.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("Saved output/figures/top15_highest_risk_metros.png")

Saved output/figures/top15_highest_risk_metros.png


/var/folders/8w/0g4j4t9j4jz3cfq3qwhrsn2r0000gn/T/ipykernel_36163/1766795072.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Calibrating the holdout risk scores

Raw XGBoost probabilities on out-of-distribution metros tend to cluster narrowly and aren't reliably interpretable as "probability of collapse." Two fixes, from simplest to most involved:

1. **Percentile rank** within the holdout population -- always safe, makes no distributional assumptions.
2. **Platt-scaled calibrated probability**, implemented as a manual 1-D logistic regression on out-of-fold predicted probabilities vs. true labels (avoids `CalibratedClassifierCV`'s metadata-routing requirements for `GroupKFold` in the installed scikit-learn version).

In [21]:
# 1. Percentile rank -- always safe.
city_risk["risk_percentile"] = city_risk["risk_score"].rank(pct=True)

# 2. Platt-scaled calibrated probability, fit on out-of-fold predicted
# probabilities from the Optuna tuning cell above (oof_probabilities).
platt_scaler = LogisticRegression()
platt_scaler.fit(oof_probabilities.reshape(-1, 1), yb_train)

holdout_scoring["risk_score_calibrated"] = platt_scaler.predict_proba(
    holdout_scoring["risk_score"].values.reshape(-1, 1)
)[:, 1]

city_risk_calibrated = (
    holdout_scoring.groupby(["cbsa", "metro_name_x"])["risk_score_calibrated"]
    .mean().reset_index()
)
city_risk = city_risk.merge(city_risk_calibrated, on=["cbsa", "metro_name_x"], how="left")
city_risk.to_csv("output/tables/holdout_city_risk_scores.csv", index=False)

print(city_risk.sort_values("risk_score", ascending=False).head(15))

     cbsa                        metro_name_x  risk_score  risk_percentile  \
0   20940                       El Centro, CA    0.988012         1.000000   
1   42140                        Santa Fe, NM    0.978919         0.997006   
2   18700                       Corvallis, OR    0.973519         0.994012   
3   41740  San Diego-Chula Vista-Carlsbad, CA    0.970378         0.991018   
4   27980                 Kahului-Wailuku, HI    0.969801         0.988024   
5   42200       Santa Maria-Santa Barbara, CA    0.969404         0.985030   
6   37100    Oxnard-Thousand Oaks-Ventura, CA    0.964629         0.982036   
7   41500                         Salinas, CA    0.963362         0.979042   
8   12700                 Barnstable Town, MA    0.963295         0.976048   
9   33540                        Missoula, MT    0.957942         0.973054   
10  13460                            Bend, OR    0.945787         0.970060   
11  13380                      Bellingham, WA    0.942294       

## Validation testing

**Leave-one-city-out cross-validation** for both models, now genuinely meaningful with 20 training cities instead of 3: `GroupKFold`, grouped by `cbsa`, with as many folds as there are training cities, holds out one full city per fold -- a direct test of the model's actual use case (scoring metros it has never seen).

Many of the newly-added cities never actually cross the affordability threshold in this window (they're "safe" negative examples), so their held-out fold has only one class present and ROC-AUC is undefined for that fold -- `np.nanmean` is used below instead of a plain mean, and the count of well-defined folds is reported alongside it.

In [22]:
f1_scorer = make_scorer(f1_score, zero_division=0)
group_cv = GroupKFold(n_splits=groups_m1.nunique())

# With 20 training cities, many held-out folds have a city that never crosses
# the affordability threshold (all one class), so ROC-AUC is undefined for
# that fold (sklearn returns NaN with an UndefinedMetricWarning). Confirmed by
# actually running this: a plain .mean() on the returned array propagates NaN
# to the whole summary, so np.nanmean/np.nanstd are used instead, and the
# count of well-defined folds is reported for transparency.

# Model 1
cv_model1 = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
f1_scores_m1 = cross_val_score(cv_model1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring=f1_scorer)
auc_scores_m1 = cross_val_score(cv_model1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="roc_auc")

n_valid_auc_m1 = np.sum(~np.isnan(auc_scores_m1))
print(f"Model 1: leave-one-city-out cross-validation ({groups_m1.nunique()} folds)")
print(f"Mean F1 score: {np.round(np.nanmean(f1_scores_m1), 3)} (std {np.round(np.nanstd(f1_scores_m1), 3)})")
print(f"Mean AUC score: {np.round(np.nanmean(auc_scores_m1), 3)} (std {np.round(np.nanstd(auc_scores_m1), 3)}) "
      f"-- {n_valid_auc_m1}/{len(auc_scores_m1)} folds had both classes present")

# Model 2
group_cv_m2 = GroupKFold(n_splits=groups_m2.nunique())
cv_model2 = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
f1_scores_m2 = cross_val_score(cv_model2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring=f1_scorer)
auc_scores_m2 = cross_val_score(cv_model2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="roc_auc")

n_valid_auc_m2 = np.sum(~np.isnan(auc_scores_m2))
print(f"\nModel 2: leave-one-city-out cross-validation ({groups_m2.nunique()} folds)")
print(f"Mean F1 score: {np.round(np.nanmean(f1_scores_m2), 3)} (std {np.round(np.nanstd(f1_scores_m2), 3)})")
print(f"Mean AUC score: {np.round(np.nanmean(auc_scores_m2), 3)} (std {np.round(np.nanstd(auc_scores_m2), 3)}) "
      f"-- {n_valid_auc_m2}/{len(auc_scores_m2)} folds had both classes present")

cv_results = pd.DataFrame({
    "model": ["model 1"] * len(f1_scores_m1) + ["model 2"] * len(f1_scores_m2),
    "held_out_fold": list(range(1, len(f1_scores_m1) + 1)) + list(range(1, len(f1_scores_m2) + 1)),
    "f1": list(f1_scores_m1) + list(f1_scores_m2),
    "roc_auc": list(auc_scores_m1) + list(auc_scores_m2)
})
cv_results.to_csv("output/tables/cv_results_final.csv", index=False)
print("\nSaved output/tables/cv_results_final.csv (roc_auc is NaN for folds with only one class present)")

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


...

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



Model 2: leave-one-city-out cross-validation (20 folds)
Mean F1 score: 0.401 (std 0.449)
Mean AUC score: 0.916 (std 0.087) -- 10/20 folds had both classes present

Saved output/tables/cv_results_final.csv (roc_auc is NaN for folds with only one class present)


## Model comparison: logistic regression baseline

A second opinion alongside XGBoost: an L2-regularized logistic regression (features standardized first), evaluated with the same leave-one-city-out `GroupKFold` used above, on the full 10-feature set and 20-city training population.

In [23]:
logit_baseline_m1 = make_pipeline(
    StandardScaler(), LogisticRegression(penalty="l2", C=1.0, max_iter=1000, class_weight="balanced")
)
f1_lr_m1 = cross_val_score(logit_baseline_m1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring=f1_scorer)
auc_lr_m1 = cross_val_score(logit_baseline_m1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="roc_auc")

logit_baseline_m2 = make_pipeline(
    StandardScaler(), LogisticRegression(penalty="l2", C=1.0, max_iter=1000, class_weight="balanced")
)
f1_lr_m2 = cross_val_score(logit_baseline_m2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring=f1_scorer)
auc_lr_m2 = cross_val_score(logit_baseline_m2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="roc_auc")

# np.nanmean, not .mean() -- see the note in the Validation Testing section
# above about single-class folds making AUC undefined.
comparison = pd.DataFrame([
    {"model": "Model 1 XGBoost", "mean_f1": np.nanmean(f1_scores_m1), "mean_auc": np.nanmean(auc_scores_m1)},
    {"model": "Model 1 Logistic Regression", "mean_f1": np.nanmean(f1_lr_m1), "mean_auc": np.nanmean(auc_lr_m1)},
    {"model": "Model 2 XGBoost", "mean_f1": np.nanmean(f1_scores_m2), "mean_auc": np.nanmean(auc_scores_m2)},
    {"model": "Model 2 Logistic Regression", "mean_f1": np.nanmean(f1_lr_m2), "mean_auc": np.nanmean(auc_lr_m2)},
])
comparison.to_csv("output/tables/model_comparison_logreg_vs_xgboost.csv", index=False)
print(comparison)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaco

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaco

                         model   mean_f1  mean_auc
0              Model 1 XGBoost  0.401467  0.915907
1  Model 1 Logistic Regression  0.411618  0.893307
2              Model 2 XGBoost  0.401467  0.915907
3  Model 2 Logistic Regression  0.411618  0.893307


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


## SHAP explainability

Full-data SHAP plots for both models (as opposed to the train-only SHAP plot in the Model 1 section above), used directly in the poster and write-up. With the full feature set and the 20-city training population, this is the best current answer to "which features actually matter."

In [24]:
# Model 1
model1_full = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
model1_full.fit(X_all, y_all)

explainer1 = shap.TreeExplainer(model1_full)
shap_values1 = explainer1.shap_values(X_all)

plt.figure()
shap.summary_plot(shap_values1, X_all, plot_type="bar", show=False)
plt.title("Model 1 -- all features, 20-city training set")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model1_full.png", dpi=150)
plt.close()

plt.figure()
shap.summary_plot(shap_values1, X_all, show=False)
plt.title("Model 1 -- all features, 20-city training set")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model1_beeswarm_full.png", dpi=150)
plt.close()

shap_importance_full_m1 = pd.DataFrame({
    "feature": ALL_FEATURES,
    "mean_abs_shap": np.abs(shap_values1).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_importance_full_m1.to_csv("output/tables/shap_importance_model1_full.csv", index=False)
print("Model 1 (all features, full data, 20 cities) SHAP importance:")
print(shap_importance_full_m1)

Model 1 (all features, full data, 20 cities) SHAP importance:
                              feature  mean_abs_shap
0                 price_to_income_lag       3.198571
3                        zhvi_qoq_lag       0.391685
4  three-year_home_price_growth_trend       0.326696
1             price_to_income_5yr_chg       0.189511
5                         hpi_yoy_lag       0.177095
6                     hpi_3yr_chg_lag       0.070081
2                        zhvi_yoy_lag       0.055378
9                        zori_yoy_lag       0.036378
8                pop_acceleration_lag       0.006302
7                    pop_velocity_lag       0.000000


In [25]:
# Model 2
model2_full = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
model2_full.fit(Xb_all, yb_all)

explainer2 = shap.TreeExplainer(model2_full)
shap_values2 = explainer2.shap_values(Xb_all)

plt.figure()
shap.summary_plot(shap_values2, Xb_all, plot_type="bar", show=False)
plt.title("Model 2 -- all features, 20-city training set")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model2_full.png", dpi=150)
plt.close()

plt.figure()
shap.summary_plot(shap_values2, Xb_all, show=False)
plt.title("Model 2 -- all features, 20-city training set")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model2_beeswarm_full.png", dpi=150)
plt.close()

shap_importance_full_m2 = pd.DataFrame({
    "feature": ALL_FEATURES,
    "mean_abs_shap": np.abs(shap_values2).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_importance_full_m2.to_csv("output/tables/shap_importance_model2_full.csv", index=False)
print("Model 2 (all features, full data, 20 cities) SHAP importance:")
print(shap_importance_full_m2)

Model 2 (all features, full data, 20 cities) SHAP importance:
                              feature  mean_abs_shap
0                 price_to_income_lag       3.198571
3                        zhvi_qoq_lag       0.391685
4  three-year_home_price_growth_trend       0.326696
1             price_to_income_5yr_chg       0.189511
5                         hpi_yoy_lag       0.177095
6                     hpi_3yr_chg_lag       0.070081
2                        zhvi_yoy_lag       0.055378
9                        zori_yoy_lag       0.036378
8                pop_acceleration_lag       0.006302
7                    pop_velocity_lag       0.000000


## Backtesting

**Leave-one-city-out** logistic-regression backtest (statsmodels `Logit`, L1-regularized): train on all-but-one training city, test on the held-out city's full history, and report AUC per held-out city. With 10 features and often only a couple dozen rows per fold, an unregularized fit hits perfect separation and a singular Hessian, so this uses `fit_regularized` instead of plain `fit`.

In [26]:
def leave_one_city_out_backtest(X_all_bt, y_all_bt, groups, city_names, label):
    aucs = {}
    for city_code in sorted(groups.unique()):
        train_mask = groups != city_code
        test_mask = groups == city_code
        y_train_bt, y_test_bt = y_all_bt[train_mask], y_all_bt[test_mask]

        if y_train_bt.nunique() < 2 or y_test_bt.nunique() < 2:
            print(f"  {label} -- {city_names.get(city_code, city_code)}: skipped (only one class present)")
            continue

        X_train_bt = X_all_bt[train_mask].copy()
        X_train_bt.insert(0, "const", 1.0)
        X_test_bt = X_all_bt[test_mask].copy()
        X_test_bt.insert(0, "const", 1.0)

        result = Logit(y_train_bt, X_train_bt).fit_regularized(method="l1", alpha=1.0, disp=0)
        pred_probs = result.predict(X_test_bt)
        auc = roc_auc_score(y_test_bt, pred_probs)
        aucs[city_code] = auc
        print(f"  {label} -- held out {city_names.get(city_code, city_code)}: AUC = {auc:.3f} (n={len(y_test_bt)})")

    if aucs:
        print(f"{label} mean leave-one-city-out AUC: {np.mean(list(aucs.values())):.3f} over {len(aucs)} cities")
    return aucs


city_names_by_cbsa = {v: k for k, v in training_cbsa_map.items()}

print("Model 1 backtest")
auc1_by_city = leave_one_city_out_backtest(X_all, y_all, groups_m1, city_names_by_cbsa, "Model 1")

Model 1 backtest
  Model 1 -- held out Austin: AUC = 0.796 (n=36)
  Model 1 -- Baltimore-Columbia-Towson, MD: skipped (only one class present)
  Model 1 -- held out Boise: AUC = 0.883 (n=36)
  Model 1 -- Buffalo-Cheektowaga, NY: skipped (only one class present)
  Model 1 -- held out Charlotte-Concord-Gastonia, NC-SC: AUC = 0.857 (n=36)
  Model 1 -- Cincinnati, OH-KY-IN: skipped (only one class present)
  Model 1 -- held out Denver-Aurora-Centennial, CO: AUC = 0.930 (n=36)
  Model 1 -- Hartford-West Hartford-East Hartford, CT: skipped (only one class present)
  Model 1 -- Indianapolis-Carmel-Greenwood, IN: skipped (only one class present)
  Model 1 -- Minneapolis-St. Paul-Bloomington, MN-WI: skipped (only one class present)
  Model 1 -- held out Nashville-Davidson--Murfreesboro--Franklin, TN: AUC = 0.935 (n=36)
  Model 1 -- held out Phoenix-Mesa-Chandler, AZ: AUC = 0.876 (n=36)
  Model 1 -- Pittsburgh, PA: skipped (only one class present)
  Model 1 -- held out Portland-Vancouver-Hillsbo

In [27]:
print("Model 2 backtest")
auc2_by_city = leave_one_city_out_backtest(Xb_all, yb_all, groups_m2, city_names_by_cbsa, "Model 2")

Model 2 backtest
  Model 2 -- held out Austin: AUC = 0.796 (n=36)
  Model 2 -- Baltimore-Columbia-Towson, MD: skipped (only one class present)
  Model 2 -- held out Boise: AUC = 0.883 (n=36)
  Model 2 -- Buffalo-Cheektowaga, NY: skipped (only one class present)
  Model 2 -- held out Charlotte-Concord-Gastonia, NC-SC: AUC = 0.857 (n=36)
  Model 2 -- Cincinnati, OH-KY-IN: skipped (only one class present)
  Model 2 -- held out Denver-Aurora-Centennial, CO: AUC = 0.930 (n=36)
  Model 2 -- Hartford-West Hartford-East Hartford, CT: skipped (only one class present)
  Model 2 -- Indianapolis-Carmel-Greenwood, IN: skipped (only one class present)
  Model 2 -- Minneapolis-St. Paul-Bloomington, MN-WI: skipped (only one class present)
  Model 2 -- held out Nashville-Davidson--Murfreesboro--Franklin, TN: AUC = 0.935 (n=36)
  Model 2 -- held out Phoenix-Mesa-Chandler, AZ: AUC = 0.876 (n=36)
  Model 2 -- Pittsburgh, PA: skipped (only one class present)
  Model 2 -- held out Portland-Vancouver-Hillsbo